
# F1 Pit-Stop Prediction with K-Nearest Neighbors (KNN)

This notebook is structured to satisfy your assignment rubric end-to-end and is easy to edit.

**Dataset:** https://www.kaggle.com/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction



## 0) Setup
If you're on Colab, upload your CSV to `/content/` or mount Drive and set `DATA_PATH` accordingly.


In [ ]:

# Uncomment in a fresh environment:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from dataclasses import dataclass
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib


## 1) Configuration

In [ ]:

# === EDIT THESE ===
DATA_PATH = "data/f1_strategy.csv"   # set to your actual CSV path
TARGET_COL = "pit_stop"              # set to your dataset's target column
K_VALUES = [3, 7, 15]                 # three different values of K as required
TEST_SIZE = 0.2
RANDOM_STATE = 42

OUTPUT_DIR = Path("outputs")
FIG_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)


## 2) Load Data + Introduction Snapshot

In [ ]:

df = pd.read_csv(DATA_PATH)
assert TARGET_COL in df.columns, f"Target column '{TARGET_COL}' not in dataset. Found: {list(df.columns)}"

n_rows, n_cols = df.shape
feature_count = n_cols - 1
numeric_count = df.drop(columns=[TARGET_COL]).select_dtypes(include=[np.number]).shape[1]
categorical_count = feature_count - numeric_count
classes = sorted(df[TARGET_COL].dropna().astype(str).unique().tolist())

print("=== INTRODUCTION SNAPSHOT ===")
print(f"Rows: {n_rows}")
print(f"Total columns: {n_cols} (features: {feature_count}, target: 1)")
print(f"Feature types -> numeric: {numeric_count}, categorical: {categorical_count}")
print(f"Number of classes in '{TARGET_COL}': {len(classes)}")
print(f"Class labels: {classes}")

df.head()


## 3) Data Discovery & Visualization

In [ ]:

# 3.1 Numeric distributions
num_df = df.select_dtypes(include=[np.number])
if not num_df.empty:
    num_df.hist(figsize=(14, 10), bins=25)
    plt.suptitle("Numeric Feature Distributions", y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "numeric_distributions.png", dpi=150)
    plt.show()

# 3.2 Correlation heatmap
if not num_df.empty:
    plt.figure(figsize=(10, 8))
    sns.heatmap(num_df.corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=150)
    plt.show()

# 3.3 PCA scatter for cluster intuition
X_tmp = df.drop(columns=[TARGET_COL])
y_tmp = df[TARGET_COL].astype(str)
X_encoded = pd.get_dummies(X_tmp, drop_first=False)
X_filled = X_encoded.fillna(X_encoded.median(numeric_only=True))

pca = PCA(n_components=2, random_state=RANDOM_STATE)
comp = pca.fit_transform(X_filled)
pca_df = pd.DataFrame({"PC1": comp[:, 0], "PC2": comp[:, 1], "target": y_tmp})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="target", alpha=0.75)
plt.title("PCA Scatter Plot (2D)")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_scatter.png", dpi=150)
plt.show()


## 4) Data Preparation (Cleaning, Encoding, Standardization)

In [ ]:

class IQRClipper(BaseEstimator, TransformerMixin):
    """Clip numeric values using IQR bounds learned from training data."""

    def fit(self, X, y=None):
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - 1.5 * iqr
        self.upper_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clip", IQRClipper()),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features),
    ])


## 5) Data Partitioning

In [ ]:

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 6, 7, 8) Train, Test, Evaluate for Different K Values

In [ ]:

@dataclass
class Result:
    k: int
    train_accuracy: float
    test_accuracy: float
    precision_macro: float
    recall_macro: float
    macro_f1: float
    misclassification_error: float

results = []
models = {}

for k in K_VALUES:
    model = Pipeline([
        ("prep", build_preprocessor(X_train)),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])

    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    precision_macro = precision_score(y_test, y_pred_test, average="macro", zero_division=0)
    recall_macro = recall_score(y_test, y_pred_test, average="macro", zero_division=0)
    macro_f1 = f1_score(y_test, y_pred_test, average="macro")

    results.append(Result(
        k=k,
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        precision_macro=precision_macro,
        recall_macro=recall_macro,
        macro_f1=macro_f1,
        misclassification_error=1 - test_acc,
    ))

    models[k] = model

    # Save per-K reports
    report = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / f"classification_report_k{k}.csv")

    labels = np.unique(np.concatenate([y_test.astype(str), y_pred_test.astype(str)]))
    cm = confusion_matrix(y_test, y_pred_test, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.to_csv(OUTPUT_DIR / f"confusion_matrix_k{k}.csv")

    # Plot confusion matrix heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix (K={k})")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"confusion_matrix_k{k}.png", dpi=150)
    plt.show()


## 9) Present the Best Model

In [ ]:

summary_df = pd.DataFrame([r.__dict__ for r in results]).sort_values(
    by=["test_accuracy", "macro_f1"],
    ascending=False,
).reset_index(drop=True)

summary_df.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)
summary_df


In [ ]:

best_k = int(summary_df.loc[0, "k"])
best_model = models[best_k]
joblib.dump(best_model, OUTPUT_DIR / "best_knn_model.joblib")

print(f"Best model selected: K={best_k}")
print("Saved to:", OUTPUT_DIR / "best_knn_model.joblib")



## 10) Conclusion (Write-up Cell)
Use this section for your final qualitative discussion:
- Which K generalized best and why?
- How did train vs test accuracy differ?
- Which classes were most frequently confused?
- What improvements could be made (feature engineering, CV, class balancing, distance metric tuning)?


## Optional: Export a compact text summary

In [ ]:

with open(OUTPUT_DIR / "auto_report.md", "w", encoding="utf-8") as f:
    f.write("# Auto-generated KNN Summary\n\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    f.write(f"Best model: K={best_k}\n")

print("Generated:", OUTPUT_DIR / "auto_report.md")
